# 🐍 Master Guide: Type Hints, Generics & Static Typing (`mypy`)
> **Single-Source Revision Guide**: Covers PEP 484 type annotations, modern union syntax (`int | str`), `Optional`, `TypeVar`, `Generic`, `Callable`, and static analysis.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Type hints allow Python code to document intent, enable rich IDE autocompletion, and prevent bugs through static type analysis tools like `mypy` or `pyright`. Type hints have **zero runtime overhead** and are not enforced by CPython at runtime.

---
## 🧠 2. CPython Architecture: The `__annotations__` Dictionary
Type hints are stored as metadata in the `__annotations__` dictionary of functions, classes, and modules.

### PEP 563 & Deferred Evaluation (`from __future__ import annotations`)
By default, type hints are evaluated at runtime. Importing `from __future__ import annotations` converts type annotations into raw strings stored in `__annotations__`, preventing circular import problems and eliminating runtime evaluation cost.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Modern Type Annotation Syntax (Python 3.10+)
Use built-in collection types directly (`list[str]`, `dict[str, int]`, and `|` for unions).

In [ ]:
# Modern Python 3.10+ syntax:
def process_scores(user_id: int, scores: list[float], fallback: str | None = None) -> dict[str, float]:
    if not scores:
        return {"average": 0.0}
    return {"average": sum(scores) / len(scores)}

print(process_scores(101, [85.5, 92.0, 78.5]))

# Inspecting runtime annotations dict:
print("\nprocess_scores.__annotations__:")
for param, hint in process_scores.__annotations__.items():
    print(f" - {param}: {hint}")

---
## 🟡 4. Part 2: Intermediate Features

### Generics & `TypeVar` (or Python 3.12+ `[T]` Syntax)
Writing functions that preserve types across arguments and returns.

In [ ]:
from typing import TypeVar, Sequence

T = TypeVar("T")

def get_first_element(items: Sequence[T]) -> T | None:
    return items[0] if items else None

print("First string :", get_first_element(["a", "b", "c"]))
print("First integer:", get_first_element([10, 20, 30]))

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Runtime Enforcement Myth
Python does NOT validate types at runtime! Passing the wrong type will not raise a `TypeError` unless checked explicitly or by static analyzers (`mypy`).

In [ ]:
def add_numbers(x: int, y: int) -> int:
    return x + y

# Notice: Passing strings succeeds without error at runtime!
print("Surprising runtime behavior:", add_numbers("hello ", "world"))
print("Rule: Type hints are static annotations, NOT runtime guardrails!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Type Hint                          | Meaning                               |
+------------------------------------+---------------------------------------+
| int | str                          | Union: Either int or str (3.10+)      |
| str | None (or Optional[str])      | Nullable string                       |
| list[int] / dict[str, float]       | Homogeneous collections               |
| Callable[[int, int], bool]         | Function signature                    |
| TypeVar('T')                       | Generic parameter                     |
| Any                                | Disables type checking                |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Enforcing Type Safety at Runtime with `@runtime_type_check`
> **Core Principle**:
> - Although Python does not natively enforce type annotations, enterprise applications can inspect `typing.get_type_hints()` to build automated runtime validation decorators for critical financial or security interfaces.

In [ ]:
# Runtime Type Checking Decorator using get_type_hints
from typing import get_type_hints
import functools

def runtime_type_check(func):
    hints = get_type_hints(func)
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Map positional arguments to parameter names
        params = list(func.__code__.co_varnames[:func.__code__.co_argcount])
        for name, arg in zip(params, args):
            expected_type = hints.get(name)
            if expected_type and not isinstance(arg, expected_type):
                raise TypeError(f"Argument '{name}' must be of type {expected_type.__name__}, got {type(arg).__name__}!")
        result = func(*args, **kwargs)
        return result
    return wrapper

@runtime_type_check
def transfer_funds(account_id: int, amount: float):
    return f"Successfully transferred Rs. {amount:,.2f} to Account #{account_id}"

print(transfer_funds(1001, 25000.0))

try:
    transfer_funds("SB-1001", 5000.0)  # Invalid account_id type
except TypeError as err:
    print("Runtime check intercepted:", err)

### 2. Structured API Contracts: `TypedDict` & `Protocol`
Defining precise dictionary schemas for JSON payloads and duck-typing structural interfaces.

In [ ]:
# TypedDict Schema & Protocol Drill
from typing import TypedDict, Protocol

class StudentPayload(TypedDict):
    sno: int
    sname: str
    course: str
    marks: float

class Serializable(Protocol):
    def serialize(self) -> dict:
        ...

def process_student_api(payload: StudentPayload) -> str:
    return f"Enrolled {payload['sname']} into {payload['course']} with Roll #{payload['sno']}"

valid_payload: StudentPayload = {"sno": 105, "sname": "Pooja", "course": "Python Full Stack", "marks": 88.5}
print("API Response:", process_student_api(valid_payload))